# 02 · Ingest Bronze — Transactions

**Layer:** Bronze &nbsp;|&nbsp; **Stage:** ingestion &nbsp;|&nbsp; **Target:** `banking_bronze.raw_transactions`

Lands the day's transaction feed. This is the high-volume side of the
pipeline and the one that carries most of the data quality risk: the upstream
switch retries on timeout, so duplicate `txn_id` values are routine, and the
merchant channel occasionally emits unnormalised currency codes and
zero-amount authorisation holds.

Runs in parallel with `01_ingest_bronze_accounts`. In the pipeline this task
is given a larger-cluster fallback because a replay day can be 50× a normal
day's volume.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType,
)

spark = SparkSession.builder.appName("medallion-bronze-transactions").enableHiveSupport().getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "16")

RUN_DATE  = "2026-09-03"
BATCH_ID  = f"batch_{RUN_DATE.replace('-', '')}"
BRONZE_DB = "banking_bronze"
SOURCE_SYSTEM = "payments-switch"
SOURCE_FILE   = f"s3a://bank-landing/transactions/dt={RUN_DATE}/txns.json"

In [ ]:
TXN_SCHEMA = StructType([
    StructField("txn_id",      StringType(), True),
    StructField("account_id",  StringType(), True),
    StructField("txn_ts",      StringType(), True),
    StructField("amount",      DoubleType(), True),
    StructField("currency",    StringType(), True),
    StructField("direction",   StringType(), True),   # DEBIT / CREDIT
    StructField("channel",     StringType(), True),   # ATM / POS / UPI / NEFT / ONLINE
    StructField("merchant",    StringType(), True),
    StructField("country",     StringType(), True),
])

rows = [
    ("TXN-90001", "ACC-1001", "2026-09-03 09:14:22",   4500.00, "INR", "DEBIT",  "UPI",    "BigBazaar",      "IN"),
    ("TXN-90002", "ACC-1001", "2026-09-03 11:02:07",  85000.00, "INR", "DEBIT",  "NEFT",   "RentPayment",    "IN"),
    ("TXN-90003", "ACC-1002", "2026-09-03 08:41:55",   1200.00, "INR", "DEBIT",  "POS",    "CafeCoffeeDay",  "IN"),
    ("TXN-90004", "ACC-1002", "2026-09-03 19:30:11",  32000.00, "INR", "CREDIT", "NEFT",   "SalaryCredit",   "IN"),
    ("TXN-90005", "ACC-1003", "2026-09-03 02:17:44",   9800.00, "USD", "DEBIT",  "ONLINE", "CloudHost",      "US"),
    ("TXN-90006", "ACC-1003", "2026-09-03 02:19:03",   9750.00, "USD", "DEBIT",  "ONLINE", "CloudHost",      "US"),
    ("TXN-90007", "ACC-1003", "2026-09-03 02:21:38",   9900.00, "USD", "DEBIT",  "ONLINE", "CloudHost",      "US"),
    ("TXN-90008", "ACC-1005", "2026-09-03 13:55:09",   2300.00, "INR", "DEBIT",  "ATM",    "ATM-HYD-1188",   "IN"),
    ("TXN-90009", "ACC-1005", "2026-09-03 14:02:47",  15000.00, "INR", "DEBIT",  "ATM",    "ATM-HYD-1188",   "IN"),
    ("TXN-90010", "ACC-1006", "2026-09-03 16:20:00",  27500.00, "EUR", "DEBIT",  "ONLINE", "LuxuryWatchCo",  "CH"),
    ("TXN-90011", "ACC-1007", "2026-09-03 10:05:31",    650.00, "INR", "DEBIT",  "UPI",    "Zomato",         "IN"),
    ("TXN-90012", "ACC-1010", "2026-09-03 21:47:12",   4100.00, "INR", "DEBIT",  "POS",    "FuelStation",    "IN"),
    ("TXN-90013", "ACC-1004", "2026-09-03 07:33:26",    300.00, "INR", "DEBIT",  "UPI",    "MilkVendor",     "IN"),
    ("TXN-90014", "ACC-1009", "2026-09-03 12:11:59",  67000.00, "INR", "DEBIT",  "NEFT",   "PropertyDeal",   "IN"),
    # --- defects: the switch retries, so duplicates are routine -----------
    ("TXN-90002", "ACC-1001", "2026-09-03 11:02:07",  85000.00, "INR", "DEBIT",  "NEFT",   "RentPayment",    "IN"),
    ("TXN-90005", "ACC-1003", "2026-09-03 02:17:44",   9800.00, "USD", "DEBIT",  "ONLINE", "CloudHost",      "US"),
    # --- defects: unnormalised / invalid values ---------------------------
    ("TXN-90015", "ACC-1002", "2026-09-03 18:00:00",   5000.00, "inr", "DEBIT",  "UPI",    "Swiggy",         "IN"),
    ("TXN-90016", "ACC-1006", "2026-09-03 15:12:30",      0.00, "EUR", "DEBIT",  "POS",    "AuthHold",       "DE"),
    ("TXN-90017", "ACC-1001", "2026-09-03 20:44:18",   -750.00, "INR", "DEBIT",  "POS",    "ReversalGlitch", "IN"),
    ("TXN-90018", "ACC-9999", "2026-09-03 22:05:41",   1800.00, "INR", "DEBIT",  "UPI",    "UnknownAcct",    "IN"),
    (None,        "ACC-1010", "2026-09-03 23:59:59",    250.00, "INR", "DEBIT",  "UPI",    "NullTxnId",      "IN"),
]

raw = spark.createDataFrame(rows, schema=TXN_SCHEMA)
print(f"landed {raw.count()} transaction records from {SOURCE_SYSTEM}")

In [ ]:
bronze_txns = (
    raw
    .withColumn("_batch_id",      F.lit(BATCH_ID))
    .withColumn("_run_date",      F.lit(RUN_DATE))
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("_source_file",   F.lit(SOURCE_FILE))
    .withColumn("_ingested_at",   F.current_timestamp())
)

(
    bronze_txns.write
    .mode("overwrite")
    .format("parquet")
    .partitionBy("_run_date")
    .saveAsTable(f"{BRONZE_DB}.raw_transactions")
)

print(f"wrote {BRONZE_DB}.raw_transactions")

In [ ]:
df = spark.table(f"{BRONZE_DB}.raw_transactions")

print("row count      :", df.count())
print("distinct txns  :", df.select("txn_id").distinct().count())
print("distinct accts :", df.select("account_id").distinct().count())

df.groupBy("channel").agg(
    F.count("*").alias("txn_count"),
    F.round(F.sum("amount"), 2).alias("total_amount"),
).orderBy(F.desc("txn_count")).show(truncate=False)